# Session Managers

Add file-based persistence to the customer service agent. Stop the agent, restart it, and watch it remember the previous conversation.

---

## Part 1: Agent Without Persistence (The Problem)

By default, agents lose their memory when you recreate them.

In [1]:
from strands import Agent, AgentSkills
from customer_service_tools import lookup_customer, get_order_history, process_refund
from dotenv import load_dotenv

SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise.

If there are previous messages in the conversation history, use that context
to continue helping the customer without asking them to repeat information."""

# First interaction
agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    system_prompt=SYSTEM_PROMPT,
)
agent("Hi, I'm customer C-1001. Can you look up my account?")

print(f"\n📝 Messages stored: {len(agent.messages)}")

Sure! Let me pull up your account right away.
Tool #1: lookup_customer
I found your account! Here's a summary:

- **Name:** Sarah Johnson
- **Email:** sarah.johnson@email.com
- **Phone:** 555-0142
- **Account Status:** Active ✅

How can I help you today, Sarah?
📝 Messages stored: 4


In [2]:
# Simulate a "restart" — create a new agent instance
agent2 = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    system_prompt=SYSTEM_PROMPT,
)

print(f"Messages after restart: {len(agent2.messages)}")
# The agent has no memory of the previous conversation!
agent2("What was my account status again?")

Messages after restart: 0
I don't have your customer information on file yet in our conversation. Could you please provide me with your **customer ID** so I can look up your account? It should be in the format **C-XXXX** (e.g., C-1001).

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "I don't have your customer information on file yet in our conversation. Could you please provide me with your **customer ID** so I can look up your account? It should be in the format **C-XXXX** (e.g., C-1001)."}], 'metadata': {'usage': {'inputTokens': 809, 'outputTokens': 59, 'totalTokens': 868}, 'metrics': {'latencyMs': 1514, 'timeToFirstByteMs': 990}}, 'tracking_id': '406eb697-9338-4041-900c-266dbd659446'}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[1.712437391281128], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='415374cc-6a86-45b0-a434-db01e198af97', usage={'inputTokens': 809, 'outputTokens': 59, 'totalTokens': 868})], usage={'inputTokens': 809, 'outputTokens': 59, 'totalTokens': 868})], traces=[<strands.telemetry.metrics.Trace object at 0xffff6d3af460>], accumulated_usage={'inputTokens': 809, 'outputTokens': 59, 'totalTokens

In [3]:
from dotenv import load_dotenv
load_dotenv()

MEMORI_URL = "https://api.memorilabs.ai/mcp/"
ENTITY_ID = "user_505"
PROCESS_ID = "my_agent"

In [4]:
import httpx2
from mcp.client.streamable_http import streamable_http_client
from strands.tools.mcp import MCPClient
import os
import sys

def create_mcp_client() -> MCPClient:
    api_key = os.getenv("MEMORI_API_KEY")
    if not api_key:
        console.print("[bold red]Error:[/] MEMORI_API_KEY not found. Add it to .env")
        sys.exit(1)

    return MCPClient(
        lambda: streamable_http_client(
            url=MEMORI_URL,
            http_client=httpx2.AsyncClient(
                headers={
                    "X-Memori-API-Key": api_key,
                    "X-Memori-Entity-Id": ENTITY_ID,
                    "X-Memori-Process-Id": PROCESS_ID,
                }
            ),
        )
    )

In [5]:
SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise.

PERSISTENT MEMORY:
- At the start of a new conversation, call your memory-recall tool to check
  if you already have a stored customer ID and account status for this
  customer before asking them to identify themselves.
- Whenever the customer provides their customer ID, or you learn their
  account status (e.g., via lookup_customer), save both to your memory tool
  right away.
- If you already have the customer ID and account status in memory, use
  them directly instead of asking again.
- Only ask for the customer ID if it isn't already in memory.

Your goal is to avoid making returning customers repeat their ID."""

In [6]:
mcp_client = create_mcp_client()

with mcp_client:
    all_mcp_tools = mcp_client.list_tools_sync() 

    memory_tools = [
        tool for tool in all_mcp_tools
        if "memori_recall" in tool.tool_name.lower() or
        "memori_advanced_augmentation" in tool.tool_name.lower()
    ]

    all_tools = [lookup_customer, get_order_history, process_refund] + memory_tools

    agent = Agent(
        tools=all_tools,
        system_prompt=SYSTEM_PROMPT,
    )
    agent("Hi, I'm customer C-1001. Can you look up my account?")

    print(f"\n📝 Messages stored: {len(agent.messages)}")


    agent2 = Agent(
        tools=all_tools,
        system_prompt=SYSTEM_PROMPT,
    )

    print(f"Messages after restart: {len(agent2.messages)}")
    # The agent has no memory of the previous conversation!
    agent2("What was my account status again?")
    

Sure! Let me look up your account and check my memory at the same time.
Tool #1: lookup_customer

Tool #2: memori_recall
I already have you on file! Here's your account summary:

- **Name:** Sarah Johnson
- **Email:** sarah.johnson@email.com
- **Phone:** 555-0142
- **Account Status:** ✅ Active

Welcome back, Sarah! How can I help you today?
📝 Messages stored: 4
Messages after restart: 0
Let me check my memory for your account information right away!
Tool #1: memori_recall
Based on my records, your account status is **Active**! Here's a quick summary of what I have on file:

- **Name:** Sarah Johnson
- **Customer ID:** C-1001
- **Email:** sarah.johnson@email.com
- **Phone:** 555-0142
- **Account Status:** Active

Is there anything else I can help you with today?